In [4]:

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [5]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()
MODEL = "gpt-4.1-mini"

OpenAI API Key exists and begins sk-proj-


In [6]:
system_message = """
You are a helpful assistant for an Airline called Flyt.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [7]:
def chat(message, history):
    history = [{"role":h["role"], "content": h["content"]} for h in history]

    message = [{"role":"system","content":system_message}] + history + [{"role": "user", "content":message}]

    responses = openai.chat.completions.create(model = MODEL, messages=message)
    return responses.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def getticketprice(destinationcity):
    print(f"Tool call city {destinationcity}")
    price = ticket_prices.get(destinationcity.lower(), "Unkown")
    return f"The price of the ticket price to the city {destinationcity} is {price}"

getticketprice("sadfsd")


Tool call city sadfsd


'The price of the ticket price to the city sadfsd is Unkown'

In [9]:
price_function = {
    "name":"getticketprice",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destinationcity":{
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destinationcity"],
        "additionalProperties": False
    }
}

In [10]:
tools = [{"type": "function", "function": price_function}] #this shit is given to openai.chat.completions.create()

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'getticketprice',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destinationcity': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destinationcity'],
    'additionalProperties': False}}}]

In [12]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)#* dont forget this tools=tools!

    # It tells you what happened at the end of the model’s output. In tool calling, when the model chooses to call a tool, the finish reason can be "tool_calls" instead of a normal completed answer. OpenAI’s docs describe tool calling as letting models interface with external systems and access data/functions provided by your app.

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)

        messages.append(message)
        messages.append(response)

        for message in messages:
            print(message)

        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

when the tool call does happen this is what we see inside the message i.e the message = response.choices[0].message

{
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_123",
            "type": "function",
            "function": {
                "name": "get_ticket_price",
                "arguments": "{\"destination_city\": \"London\"}"
            }
        }
    ]
}

we are saving the assistant tools request response(this is not the final answer just a pivot)


In [13]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "getticketprice":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destinationcity')
        price_details = getticketprice(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [15]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def getticketprice(destinationcity):
    print(f"Tool call city {destinationcity}")
    price = ticket_prices.get(destinationcity.lower(), "Unkown")
    return f"The price of the ticket price to the city {destinationcity} is {price}"

In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)#* dont forget this tools=tools!

    # It tells you what happened at the end of the model’s output. In tool calling, when the model chooses to call a tool, the finish reason can be "tool_calls" instead of a normal completed answer. OpenAI’s docs describe tool calling as letting models interface with external systems and access data/functions provided by your app.

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)

        messages.append(message)
        messages.extend(response) #* imp

        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [17]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "getticketprice":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destinationcity")
            price_details = getticketprice(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [18]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [32]:
def chat(message, history):
    history = [{"role": r["role"], "content":r['content']} for r in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [20]:
gr.ChatInterface(fn=chat, type= "messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [21]:
import sqlite3

In [22]:
DB = "prices.db"

In [23]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [24]:
def get_ticket_price(city):
    print(f" DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else f"No price data available for {city}"

In [25]:
get_ticket_price("London")

 DATABASE TOOL CALLED: Getting price for London


'No price data available for London'

In [26]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?,?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items(): # enumerate() ante key index, key value ostadi ra rei
    set_ticket_price(city, price)

In [29]:
get_ticket_price('Tokyo')

 DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tool call city Hong Kong
Tool call city sydney


In [34]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?,?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

    return f"price of {city} has been set to {price}"

In [65]:
get_price_function = {
    "name":"get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city":{
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name":"set_ticket_price",
    "description": "Set the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city":{
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The price of the ticket to that city, the customer wants to travel to",
            }
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": get_price_function},{"type": "function", "function": set_price_function}] 

In [62]:
def chat(message, history):
    history = [{"role": r["role"], "content":r['content']} for r in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [66]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            price_details = get_ticket_price(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        
        elif tool_call.function.name=="set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            price = arguments.get("price")
            price_details = set_ticket_price(city, price)
            
    return responses


In [67]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


 DATABASE TOOL CALLED: Getting price for Tokyo


In [44]:
get_ticket_price("Tokyo")

 DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $420.0'

In [56]:
gr.ChatInterface(fn= chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool call city Tokyo
